<a href="https://colab.research.google.com/github/warry258/colab/blob/main/%E8%87%AA%E5%8A%A8%E6%89%93%E8%BD%B41_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 字幕自动打轴工具 — Forced Alignment for SRT Generation

使用 HuggingFace 强制对齐模型将音频与文本对齐，自动生成带精准时间轴的 SRT 字幕文件。

## 支持模型
- **Qwen/Qwen3-ForcedAligner-0.6B** — 支持 11 种语言，需要 GPU
- **deskpai/ctc_forced_aligner** — 基于 CTC，CPU/GPU 均可运行

## 核心功能
- **鲁棒文本匹配**: 自动忽略标点符号和空格，精确将模型输出合并到预分段短句
- **双后端**: CPU 和 GPU 环境自动适配
- **SRT 输出**: 生成标准字幕格式，可直接导入视频编辑软件

In [ ]:
#@title 1. 安装依赖 { display-mode: "form" }

import sys
import subprocess

def pip_install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", package])

def pip_uninstall(package):
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", package])

print("正在安装依赖包...")

# 彻底清理可能冲突的 CTC 包
for pkg in ["ctc_forced_aligner", "ctc-forced-aligner"]:
    try:
        pip_uninstall(pkg)
    except Exception:
        pass

# HuggingFace 核心（确保版本兼容）
pip_install("transformers>=4.34")

# Gradio UI
pip_install("gradio")

# 音频处理
pip_install("soundfile")

# CTC Forced Aligner — 从 GitHub 源码安装 (MahmoudAshraf97)
pip_install("git+https://github.com/MahmoudAshraf97/ctc-forced-aligner.git")

# Qwen ASR — GPU only
try:
    pip_install("qwen-asr")
    print(" qwen-asr 安装成功")
except Exception as e:
    print(f" qwen-asr 安装失败: {e}")

print("\n依赖安装完成!")
print("如遇到 import 错误，请: 运行时 → 重启运行时 → 重新运行所有单元格")


In [ ]:
#@title 2. 导入库与设备检测

import os
import re
import sys
import json
import tempfile
import unicodedata
from pathlib import Path
from typing import List, Tuple, Dict, Optional

import torch
import gradio as gr

# ---- 设备检测 ----
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_AVAILABLE = torch.cuda.is_available()

print(f"运行设备: {DEVICE}")
print(f"GPU 可用: {GPU_AVAILABLE}")
if GPU_AVAILABLE:
    print(f"GPU 型号: {torch.cuda.get_device_name(0)}")
    print(f"GPU 显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# ---- 模型可用性检测 ----
QWEN_AVAILABLE = False
try:
    import qwen_asr
    QWEN_AVAILABLE = True
    print(" Qwen ASR 已就绪")
except ImportError:
    print(" Qwen ASR 不可用（安装失败或环境不兼容）")

CTC_AVAILABLE = False
try:
    import ctc_forced_aligner
    CTC_AVAILABLE = True
    print(" CTC Forced Aligner 已就绪")
except ImportError:
    print(" CTC Forced Aligner 不可用")

if not QWEN_AVAILABLE and not CTC_AVAILABLE:
    raise RuntimeError("无可用模型！请重新运行安装单元格。")


In [ ]:
#@title 3. 核心算法：鲁棒字符计数匹配 + SRT 格式化

import re


def get_pure_text_length(text: str) -> int:
    """
    计算"纯净"字符数：去除所有标点、空格、控制字符后剩余的字符数。
    保留字母、数字、CJK 字符、日文假名。
    """
    return len(re.sub(
        r'[^\w一-鿿぀-ゟ゠-ヿ]',
        '', str(text)
    ).lower())


def merge_token_timestamps_to_sentences(
    token_timestamps: List[Tuple[str, float, float]],
    target_sentences: List[str],
    debug: bool = False
) -> List[Dict]:
    """
    鲁棒合并：通过字符数累计将模型输出的词/字级时间戳匹配到预分段短句。

    原理（来自 Gemini 验证过的方案）：
    - 强制对齐保证 token 顺序 === 句子顺序
    - 对每个句子：累计 token 的"纯净"字符数，直到填满该句的目标字符数
    - 第一个 token 的 start_time = 句子开始时间
    - 最后一个 token 的 end_time = 句子结束时间
    - 标点 token 纯净字符数为 0，自动被跳过

    Args:
        token_timestamps: [(text, start_time, end_time), ...]  模型输出的 token 级时间戳
        target_sentences: [str, ...]  用户预分段的短句列表
        debug: 是否打印调试信息

    Returns:
        [{"text": str, "start": float, "end": float}, ...]
    """
    if not target_sentences:
        return []

    results = []
    token_idx = 0
    total_tokens = len(token_timestamps)

    for sent_idx, sentence in enumerate(target_sentences):
        t_len = get_pure_text_length(sentence)
        if t_len == 0:
            results.append({"text": sentence, "start": 0.0, "end": 0.0})
            continue

        acc_len = 0
        st, et = None, None

        while token_idx < total_tokens and acc_len < t_len:
            seg_text, seg_start, seg_end = token_timestamps[token_idx]
            if st is None:
                st = seg_start
            et = seg_end
            acc_len += get_pure_text_length(seg_text)
            token_idx += 1

        if debug and sent_idx < 5:
            print(f"  [{sent_idx}] \"{sentence[:50]}\" -> "
                  f"tokens[{token_idx - (acc_len and 1 or 0)}:{token_idx}] "
                  f"char_cnt={acc_len}/{t_len} "
                  f"time={st:.2f}s-{et:.2f}s" if st else "")

        if st is not None and et is not None:
            results.append({
                "text": sentence,
                "start": round(st, 3),
                "end": round(et, 3),
            })
        else:
            results.append({"text": sentence, "start": 0.0, "end": 0.0})

    # ---- 后处理：修复缺失/异常时间戳 ----
    for i in range(len(results)):
        if results[i]["start"] == 0.0 and results[i]["end"] == 0.0:
            for j in range(i - 1, -1, -1):
                if results[j]["end"] > 0:
                    results[i]["start"] = results[j]["end"]
                    results[i]["end"] = results[j]["end"]
                    break
            if results[i]["start"] == 0.0:
                for j in range(i + 1, len(results)):
                    if results[j]["start"] > 0:
                        results[i]["start"] = results[j]["start"]
                        results[i]["end"] = results[j]["start"]
                        break

    for i in range(len(results)):
        if i > 0 and results[i]["start"] < results[i - 1]["end"]:
            results[i]["start"] = results[i - 1]["end"]
        if results[i]["end"] < results[i]["start"]:
            results[i]["end"] = results[i]["start"] + 0.001

    if debug:
        non_zero = sum(1 for r in results if r["start"] > 0 or r["end"] > 0)
        print(f"时间戳覆盖率: {non_zero}/{len(results)} 句")

    return results


def seconds_to_srt_time(seconds: float) -> str:
    seconds = max(0, seconds)
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    ms = int((seconds % 1) * 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


def format_srt(segments: List[Dict]) -> str:
    lines = []
    index = 1
    for seg in segments:
        text = seg["text"].strip()
        if not text:
            continue
        lines.append(str(index))
        lines.append(
            f"{seconds_to_srt_time(seg['start'])} --> {seconds_to_srt_time(seg['end'])}"
        )
        lines.append(text)
        lines.append("")
        index += 1
    return "\n".join(lines)


print("核心算法已加载")


In [ ]:
#@title 4. Qwen3-ForcedAligner 模型封装

import tempfile
import soundfile as sf


def run_qwen_alignment(
    audio_path: str,
    full_text: str,
    target_sentences: List[str],
    language: str = "Chinese"
) -> List[Dict]:
    """
    使用 Qwen3-ForcedAligner-0.6B 进行强制对齐。
    采用 Gemini 验证过的滑动窗口切片方案，支持任意时长音频。
    """
    from qwen_asr import Qwen3ForcedAligner

    print("🚀 加载 Qwen3-ForcedAligner-0.6B ...")
    model = Qwen3ForcedAligner.from_pretrained(
        "Qwen/Qwen3-ForcedAligner-0.6B",
        dtype=torch.bfloat16,
        device_map="cuda:0",
    )

    # 读取完整音频
    audio_data, sr = sf.read(audio_path)
    total_duration = len(audio_data) / sr
    print(f"📊 音频总时长: {total_duration:.1f}s")

    # 切片参数（与 Gemini 一致）
    MAX_CHUNK_DUR = 240.0       # 每次最多 4 分钟
    SAFE_TAIL_MARGIN = 15.0     # 丢弃 chunk 末尾 15s 的不完整句子

    remaining = list(target_sentences)
    time_offset = 0.0
    all_segments = []
    chunk_idx = 0

    while remaining and time_offset < total_duration:
        chunk_idx += 1
        chunk_dur = min(MAX_CHUNK_DUR, total_duration - time_offset)
        is_last = (time_offset + chunk_dur >= total_duration - 1.0)

        # 切割音频
        start_f = int(time_offset * sr)
        end_f = int((time_offset + chunk_dur) * sr)
        chunk_audio = audio_data[start_f:end_f]

        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as f:
            sf.write(f.name, chunk_audio, sr)
            chunk_path = f.name

        # 对齐：将所有剩余文本喂给模型
        chunk_text = " ".join(remaining)
        print(f"\n▶️ Chunk {chunk_idx}: 音频[{time_offset:.0f}s-{time_offset + chunk_dur:.0f}s] "
              f"剩余{len(remaining)}句")

        results = model.align(audio=chunk_path, text=chunk_text, language=language)
        tokens = results[0]  # List[AlignmentResult]

        # 提取 (text, start, end) 元组
        token_data = []
        for seg in tokens:
            try:
                token_data.append((seg.text, seg.start_time, seg.end_time))
            except AttributeError:
                # 兼容不同版本 qwen-asr
                d = vars(seg) if hasattr(seg, '__dict__') else {}
                token_data.append((
                    d.get('text', d.get('token', d.get('word', ''))),
                    d.get('start_time', d.get('start', 0.0)),
                    d.get('end_time', d.get('end', 0.0)),
                ))

        # 用字符计数法匹配句子（Gemini 方案）
        matched = []
        ti = 0
        for sentence in remaining:
            t_len = get_pure_text_length(sentence)
            if t_len == 0:
                continue
            acc = 0
            st, et = None, None
            while ti < len(token_data) and acc < t_len:
                seg_text, seg_start, seg_end = token_data[ti]
                if st is None:
                    st = seg_start
                et = seg_end
                acc += get_pure_text_length(seg_text)
                ti += 1
            if st is not None and et is not None:
                matched.append({"text": sentence, "start": st, "end": et})

        # 寻找安全切分点（丢弃 chunk 末尾被截断的句子）
        if is_last:
            valid = matched
            remaining = []
        else:
            valid_idx = -1
            for i, m in enumerate(matched):
                if m["end"] < (chunk_dur - SAFE_TAIL_MARGIN):
                    valid_idx = i
                else:
                    break
            if valid_idx == -1 and len(matched) > 0:
                valid_idx = 0  # 兜底：至少保留第一句
            valid = matched[:valid_idx + 1]
            remaining = remaining[valid_idx + 1:]

        print(f"  本段对齐 {len(valid)} 句（共{len(matched)}句匹配）")

        # 写入结果（加时间偏移）
        for m in valid:
            all_segments.append({
                "text": m["text"],
                "start": round(m["start"] + time_offset, 3),
                "end": round(m["end"] + time_offset, 3),
            })

        # 下一轮起点
        if valid:
            time_offset = time_offset + valid[-1]["end"]
        else:
            time_offset = total_duration

        os.unlink(chunk_path)
        torch.cuda.empty_cache()

    del model
    torch.cuda.empty_cache()

    print(f"\n✅ Qwen 对齐完成：{len(all_segments)} 句")
    return all_segments


print("Qwen 模型封装已加载")


In [ ]:
#@title 5. CTC Forced Aligner 模型封装

def run_ctc_alignment(
    audio_path: str,
    full_text: str,
    target_sentences: List[str],
    language: str = "eng"
) -> List[Dict]:
    """
    使用 CTC Forced Aligner 进行强制对齐。
    修复了词级别向字符级别对齐时的零帧 BUG，获取准确时间戳。
    """
    from ctc_forced_aligner import (
        load_audio,
        load_alignment_model,
        generate_emissions,
        preprocess_text,
        get_alignments,
    )
    import ctc_forced_aligner.alignment_utils as ctc_au
    import ctc_forced_aligner.text_utils as ctc_tu

    dtype = torch.float16 if DEVICE == "cuda" else torch.float32

    # ---- 容错补丁开始 ----
    _original_get_spans = ctc_au.get_spans
    _original_postprocess = ctc_tu.postprocess_results

    def _relaxed_get_spans(tokens_starred, segments, blank_token):
        n_seg = len(segments)
        spans =[]
        si = 0

        for ti, token in enumerate(tokens_starred):
            # 核心修复：把 "h e l l o" 拆成["h", "e", "l", "l", "o"]
            target_letters = token.split(" ")

            # 跳过该 token 前面的空白帧
            while si < n_seg and segments[si].label == blank_token:
                si += 1

            start_seg_idx = si
            end_seg_idx = si
            matched_any = False

            # 逐个字符匹配时间帧
            for ltr in target_letters:
                while si < n_seg and segments[si].label == blank_token:
                    si += 1

                # 如果当前字符匹配成功，记录帧索引并推进
                if si < n_seg and segments[si].label == ltr:
                    if not matched_any:
                        start_seg_idx = si
                    end_seg_idx = si
                    matched_any = True
                    si += 1

            if not matched_any:
                # 兜底：如果整个词都没在音频里找到（可能被模型吞字），给个安全索引
                safe_idx = min(start_seg_idx, n_seg - 1) if n_seg > 0 else 0
                spans.append([ctc_au.Segment(token, safe_idx, safe_idx)])
            else:
                # 截取该词从第一个字母到最后一个字母的所有帧
                span = segments[start_seg_idx : end_seg_idx + 1]
                spans.append(span)

        return spans

    def _safe_postprocess_results(text_starred, spans, stride, scores, merge_threshold=0.0):
        results =[]
        for i, t in enumerate(text_starred):
            if t == "<star>":
                continue
            span = spans[i]
            if not span:
                continue

            seg_start_idx = span[0].start
            seg_end_idx = span[-1].end

            # 转换为秒 (1000ms = 1s)
            audio_start_sec = seg_start_idx * stride / 1000.0
            audio_end_sec = seg_end_idx * stride / 1000.0

            if seg_end_idx >= seg_start_idx:
                score = scores[seg_start_idx : seg_end_idx + 1].sum()
            else:
                score = 0.0

            score_val = score.item() if hasattr(score, "item") else float(score)

            sample = {
                "start": audio_start_sec,
                "end": audio_end_sec,
                "text": t,
                "score": score_val,
            }
            results.append(sample)

        ctc_tu.merge_segments(results, merge_threshold)
        return results

    # 应用补丁
    ctc_au.get_spans = _relaxed_get_spans
    ctc_tu.postprocess_results = _safe_postprocess_results
    # ---- 容错补丁结束 ----

    print(f"🚀 加载 CTC 对齐模型: MahmoudAshraf/mms-300m-1130-forced-aligner (设备: {DEVICE})")
    alignment_model, alignment_tokenizer = load_alignment_model(
        DEVICE,
        dtype=dtype,
    )

    print("🔄 加载音频...")
    audio_waveform = load_audio(
        audio_path, alignment_model.dtype, alignment_model.device
    )

    print("🔄 生成发射矩阵...")
    emissions, stride = generate_emissions(
        alignment_model, audio_waveform, batch_size=8
    )

    print("🔄 预处理文本...")
    non_latin = {"cmn", "zho", "chi", "jpn", "ja", "kor", "ko", "ara", "ar", "rus", "ru"}
    needs_romanize = language in non_latin
    tokens_starred, text_starred = preprocess_text(
        full_text,
        romanize=needs_romanize,
        language=language,
    )

    print("🔄 CTC 解码...")
    segments_raw, scores, blank_token = get_alignments(
        emissions, tokens_starred, alignment_tokenizer
    )

    print("🔄 获取时间跨度 (容错模式)...")
    spans = ctc_au.get_spans(tokens_starred, segments_raw, blank_token)

    print("🔄 后处理 (容错模式)...")
    results = ctc_tu.postprocess_results(text_starred, spans, stride, scores)

    token_timestamps =[]
    for seg in results:
        token_timestamps.append((seg["text"], seg["start"], seg["end"]))

    print(f"模型输出 {len(token_timestamps)} 个词/字级时间戳")

    # 引用前文的方法匹配句子
    segments = merge_token_timestamps_to_sentences(
        token_timestamps, target_sentences, debug=True
    )

    # 恢复原函数，防止污染全局环境
    ctc_au.get_spans = _original_get_spans
    ctc_tu.postprocess_results = _original_postprocess

    del alignment_model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    return segments

print("CTC 模型封装已加载")

In [ ]:
#@title 6. 主处理流程

def process_alignment(
    audio_file,
    text_input: str,
    text_file,
    model_choice: str,
    language: str
):
    """
    主处理函数，供 Gradio UI 调用。
    """
    debug_lines = []

    # ---- 验证输入 ----
    if audio_file is None:
        return "", "请上传音频文件", "", None

    # 获取文本（优先上传文件，其次文本框）
    raw_text = ""
    if text_file is not None:
        try:
            file_path = None
            if isinstance(text_file, str):
                file_path = text_file
            elif isinstance(text_file, dict):
                file_path = text_file.get("name", "")
            elif hasattr(text_file, "name"):
                file_path = text_file.name
            else:
                file_path = str(text_file)

            if file_path and os.path.exists(file_path):
                with open(file_path, "r", encoding="utf-8") as f:
                    raw_text = f.read()
                debug_lines.append(f"从文件读取文本 ({len(raw_text)} 字符)")
            else:
                debug_lines.append(f"无法读取文本文件: {file_path}")
        except Exception as e:
            debug_lines.append(f"读取文本文件失败: {e}")
            raw_text = ""

    if not raw_text and text_input:
        raw_text = text_input

    if not raw_text or not raw_text.strip():
        return "", "请输入文本或上传文本文件", "", None

    # 解析为句子列表
    target_sentences = [
        line.strip() for line in raw_text.strip().splitlines() if line.strip()
    ]

    if not target_sentences:
        return "", "文本为空或格式不正确（需要每行一个短句）", "", None

    full_text = " ".join(target_sentences)

    debug_lines += [
        f"音频文件: {audio_file}",
        f"模型选择: {model_choice}",
        f"语言: {language}",
        f"目标句子数: {len(target_sentences)}",
        f"总字符数: {len(full_text)}",
        "",
    ]

    # ---- 运行对齐 ----
    try:
        if "Qwen" in model_choice:
            if not GPU_AVAILABLE:
                return "", "Qwen 模型需要 GPU，当前环境无 GPU 可用。请切换到 CTC 模型。", "", None
            if not QWEN_AVAILABLE:
                return "", "Qwen ASR 未安装。请重新运行安装单元格。", "", None

            lang_map = {
                "中文": "Chinese", "英文": "English", "日语": "Japanese",
                "韩语": "Korean", "法语": "French", "德语": "German",
                "俄语": "Russian", "西班牙语": "Spanish", "意大利语": "Italian",
                "葡萄牙语": "Portuguese", "粤语": "Cantonese",
            }
            lang = lang_map.get(language, "Chinese")
            debug_lines.append(f"Qwen 语言代码: {lang}")

            segments = run_qwen_alignment(audio_file, full_text, target_sentences, lang)
        else:
            if not CTC_AVAILABLE:
                return "", "CTC Forced Aligner 未安装。请重新运行安装单元格。", "", None

            lang_map = {
                "中文": "cmn", "英文": "eng", "日语": "jpn",
                "韩语": "kor", "法语": "fra", "德语": "deu",
                "俄语": "rus", "西班牙语": "spa", "意大利语": "ita",
                "葡萄牙语": "por",
            }
            lang = lang_map.get(language, "cmn")
            debug_lines.append(f"CTC 语言代码: {lang}")

            segments = run_ctc_alignment(audio_file, full_text, target_sentences, lang)

        # ---- 生成 SRT ----
        srt_content = format_srt(segments)

        debug_lines.append("")
        debug_lines.append(f"对齐完成! 共 {len(segments)} 段")
        for seg in segments[:15]:
            debug_lines.append(
                f"  [{seg['start']:.2f}s - {seg['end']:.2f}s] {seg['text'][:60]}"
            )
        if len(segments) > 15:
            debug_lines.append(f"  ... 共 {len(segments)} 段")

        status = f"对齐完成! 共 {len(segments)} 段"

        srt_path = None
        if srt_content:
            tmp = tempfile.NamedTemporaryFile(
                mode="w", suffix=".srt", delete=False, encoding="utf-8"
            )
            tmp.write(srt_content)
            tmp.close()
            srt_path = tmp.name

        return srt_content, status, "\n".join(debug_lines), srt_path

    except Exception as e:
        import traceback
        error_detail = traceback.format_exc()
        debug_lines.append(f"\n错误: {e}\n\n{error_detail}")
        return "", f"处理出错: {str(e)}", "\n".join(debug_lines), None


print("主处理流程已加载")


In [ ]:
#@title 7. 启动 Gradio 界面

# 构建可用模型列表
model_choices = []
if CTC_AVAILABLE:
    model_choices.append("CTC Forced Aligner (CPU/GPU)")
if QWEN_AVAILABLE and GPU_AVAILABLE:
    model_choices.append("Qwen3-ForcedAligner (GPU)")

if not model_choices:
    print("错误: 无可用模型!")
else:
    with gr.Blocks(title="字幕自动打轴工具") as demo:

        gr.Markdown("""
        # 字幕自动打轴工具

        **将音频与文本自动对齐，生成带精准时间轴的 SRT 字幕文件。**

        ---

        ### 使用步骤
        1. **上传音频文件** (wav / mp3 / m4a / flac)
        2. **输入文本** — 每行一个短句，或上传 .txt 文本文件
        3. **选择对齐模型** 和 **语言**
        4. 点击 **"开始对齐"** 按钮
        5. 查看生成的 SRT 并下载

        ---
        """)

        with gr.Row():
            with gr.Column(scale=2):
                gr.Markdown("### 输入")

                audio_input = gr.Audio(
                    label="音频文件",
                    type="filepath",
                )

                text_input = gr.Textbox(
                    label="文本内容（每行一个短句）",
                    placeholder="今天天气真好。\n我们一起去公园吧。\n你准备好了吗？",
                    lines=8,
                    max_lines=20,
                )

                text_file = gr.File(
                    label="或上传文本文件 (.txt)",
                    file_types=[".txt"],
                )

                with gr.Row():
                    model_choice = gr.Dropdown(
                        label="对齐模型",
                        choices=model_choices,
                        value=model_choices[0],
                        scale=1,
                    )
                    language_choice = gr.Dropdown(
                        label="音频语言",
                        choices=[
                            "中文", "英文", "日语", "韩语",
                            "法语", "德语", "俄语", "西班牙语",
                            "意大利语", "葡萄牙语",
                        ],
                        value="中文",
                        scale=1,
                    )

                submit_btn = gr.Button(
                    "开始对齐",
                    variant="primary",
                    size="lg",
                )

                status_output = gr.Textbox(
                    label="状态",
                    value="等待输入...",
                    interactive=False,
                )

            with gr.Column(scale=2):
                gr.Markdown("### 输出")

                srt_output = gr.Textbox(
                    label="生成的 SRT 字幕",
                    lines=18,
                    max_lines=30,
                    interactive=False,
                    elem_classes=["srt-output"],
                )

                srt_download = gr.File(
                    label="下载 SRT 文件",
                    interactive=False,
                )

                with gr.Accordion("调试信息", open=False):
                    debug_output = gr.Textbox(
                        label="详细日志",
                        lines=12,
                        max_lines=20,
                        interactive=False,
                    )

        # 绑定事件
        submit_btn.click(
            fn=process_alignment,
            inputs=[
                audio_input,
                text_input,
                text_file,
                model_choice,
                language_choice,
            ],
            outputs=[srt_output, status_output, debug_output, srt_download],
        )

    # 启动 (Gradio 6.0: theme/css 参数移到 launch)
    demo.queue(max_size=5).launch(
        share=True,
        debug=False,
        show_error=True,
        theme="soft",
        css="""
        .srt-output textarea { font-family: "Courier New", monospace; font-size: 13px; }
        footer { visibility: hidden; }
        """,
    )
